# Titanic Survival Prediction

Beginner-friendly classification project for Google Colab. Covers CSV loading, EDA, missing-value inspection, KNN imputation, categorical imputation, one-hot encoding, train-test split, scikit-learn pipelines, logistic regression, and evaluation. No interview questions are included in this project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load the CSV

The dataset is downloaded from a public GitHub CSV source directly into the Colab runtime.

In [ ]:
csv_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(csv_url)
print("Shape:", df.shape)
display(df.head())
df.info()

## 2. Exploratory data analysis

Inspect target counts, age distribution, and survival patterns.

In [ ]:
display(df.describe(include="all").T)
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df["Survived"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Survived", ax=axes[0])
axes[0].set_title("Survival counts (0 = No, 1 = Yes)")
sns.histplot(data=df, x="Age", bins=25, kde=True, ax=axes[1])
axes[1].set_title("Age distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="Pclass", y="Survived", errorbar=None)
plt.title("Survival rate by passenger class")
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="Sex", y="Survived", errorbar=None)
plt.title("Survival rate by sex")
plt.show()

## 3. Select features and split data

We use a small set of understandable features and keep the target separate. Columns such as Cabin, Name, Ticket, and PassengerId are excluded to keep the project focused.

In [ ]:
features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]
X = df[features].copy()
y = df["Survived"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 4. Preprocessing

Numerical values are standardized and missing values are estimated using KNNImputer. Categorical missing values are filled with the most frequent category, then one-hot encoded. The pipeline is fitted only on training data to avoid test-data leakage.

In [ ]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=5))
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 5. Train logistic regression

Logistic regression is a classification algorithm. Here it predicts whether a passenger survived (1) or did not survive (0).

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
print("Training complete.")

## 6. Evaluate the model

Accuracy is the proportion of test predictions that are correct. The confusion matrix and classification report provide more detail about errors.

In [ ]:
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")

cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(
    cm,
    index=["Actual did not survive", "Actual survived"],
    columns=["Predicted did not survive", "Predicted survived"]
))
print(classification_report(y_test, y_pred, zero_division=0))

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 7. Try one sample prediction

Edit the values to test a hypothetical passenger. This is a model prediction, not a statement about a real person.

In [ ]:
sample_passenger = pd.DataFrame([{
    "Pclass": 3,
    "Age": 22,
    "SibSp": 1,
    "Parch": 0,
    "Fare": 7.25,
    "Sex": "male",
    "Embarked": "S"
}])
prediction = model.predict(sample_passenger)[0]
survival_probability = model.predict_proba(sample_passenger)[0][1]
print("Prediction:", "Survived" if prediction == 1 else "Did not survive")
print(f"Estimated survival probability: {survival_probability:.3f}")

## Conclusion and limitations

This project demonstrates a complete beginner-level classification workflow. The data is historical, accuracy alone does not explain every kind of error, and predictions should not be treated as causal explanations.

Dataset source: https://github.com/datasciencedojo/datasets

Run this notebook in Google Colab by opening Colab and choosing File → Upload notebook.